In [ ]:
# ==========================================================
# COVID TRAINING
# SECTION 1 : MOUNT GOOGLE DRIVE
# ==========================================================

from google.colab import drive

drive.mount('/content/drive')

print("="*60)
print("GOOGLE DRIVE CONNECTED SUCCESSFULLY")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GOOGLE DRIVE CONNECTED SUCCESSFULLY


In [ ]:
# ==========================================================
# SECTION 2 : IMPORT LIBRARIES
# ==========================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("="*60)
print("LIBRARIES IMPORTED SUCCESSFULLY")
print("="*60)

LIBRARIES IMPORTED SUCCESSFULLY


In [ ]:
# ==========================================================
# SECTION 3 : DATASET PATH
# ==========================================================

import os

DATASET_PATH = "/content/drive/MyDrive/Research_paper/Covid_Data_set"

TRAIN_PATH = os.path.join(DATASET_PATH, "Covid_Split", "train")
VALID_PATH = os.path.join(DATASET_PATH, "Covid_Split", "validation")
TEST_PATH  = os.path.join(DATASET_PATH, "Covid_Split", "test")

print("="*60)
print("DATASET PATH VERIFIED")
print("="*60)

print("Train Exists      :", os.path.exists(TRAIN_PATH))
print("Validation Exists :", os.path.exists(VALID_PATH))
print("Test Exists       :", os.path.exists(TEST_PATH))

DATASET PATH VERIFIED
Train Exists      : True
Validation Exists : True
Test Exists       : True


In [37]:
# ==========================================================
# SECTION 4 : HYPERPARAMETERS
# ==========================================================

# Image Size
IMAGE_SIZE = (224, 224)

# Batch Size (Base Paper)
BATCH_SIZE = 60

# Number of Epochs (Base Paper)
EPOCHS = 100

# Number of Classes
NUM_CLASSES = 4

print("="*60)
print("HYPERPARAMETERS")
print("="*60)
print("Image Size      :", IMAGE_SIZE)
print("Batch Size      :", BATCH_SIZE)
print("Epochs          :", EPOCHS)
print("Number of Class :", NUM_CLASSES)
print("="*60)

HYPERPARAMETERS
Image Size      : (224, 224)
Batch Size      : 60
Epochs          : 100
Number of Class : 4


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'server  final.sln', 'Research_paper']


In [ ]:
import os

print("="*60)
print("Research_paper Folder")
print("="*60)

print(os.listdir("/content/drive/MyDrive/Research_paper"))

Research_paper Folder
['Covid_Data_set']


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/Research_paper/Covid_Data_set"))

['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia', 'Covid_Split']


In [38]:
# ==========================================================
# SECTION 5 : IMAGE DATA GENERATORS
# ==========================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Training Data
train_datagen = ImageDataGenerator(
    rescale=1./255
)

# Validation Data
valid_datagen = ImageDataGenerator(
    rescale=1./255
)

# Testing Data
test_datagen = ImageDataGenerator(
    rescale=1./255
)

# -----------------------------
# Training Generator
# -----------------------------
train_generator = train_datagen.flow_from_directory(

    TRAIN_PATH,

    target_size=IMAGE_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

# -----------------------------
# Validation Generator
# -----------------------------
valid_generator = valid_datagen.flow_from_directory(

    VALID_PATH,

    target_size=IMAGE_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False

)

# -----------------------------
# Test Generator
# -----------------------------
test_generator = test_datagen.flow_from_directory(

    TEST_PATH,

    target_size=IMAGE_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False

)

print("="*60)
print("DATA GENERATORS CREATED SUCCESSFULLY")
print("="*60)
print("Training Images   :", train_generator.samples)
print("Validation Images :", valid_generator.samples)
print("Testing Images    :", test_generator.samples)
print("Batch Size        :", train_generator.batch_size)
print("Steps Per Epoch   :", len(train_generator))
print("="*60)

Found 14814 images belonging to 4 classes.
Found 3175 images belonging to 4 classes.
Found 3176 images belonging to 4 classes.
DATA GENERATORS CREATED SUCCESSFULLY
Training Images   : 14814
Validation Images : 3175
Testing Images    : 3176
Batch Size        : 60
Steps Per Epoch   : 247


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'server  final.sln', 'Research_paper']


In [ ]:
import os

for item in os.scandir("/content/drive/MyDrive"):
    print(item.name)
    print("Directory :", item.is_dir())
    print("Symlink   :", item.is_symlink())
    print("-"*50)

Colab Notebooks
Directory : True
Symlink   : False
--------------------------------------------------
server  final.sln
Directory : False
Symlink   : False
--------------------------------------------------
Research_paper
Directory : True
Symlink   : False
--------------------------------------------------


In [ ]:
import os

ROOT = "/content/drive/MyDrive"

for root, dirs, files in os.walk(ROOT):
    print(root)

/content/drive/MyDrive
/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Research_paper
/content/drive/MyDrive/Research_paper/Models


In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/Research_paper/Models"

print("="*60)
print("MODELS FOLDER")
print("="*60)

print(os.listdir(MODEL_DIR))

MODELS FOLDER
['covid_vgg16_base_model.keras']


In [ ]:
from tensorflow.keras.applications import VGG16

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

print("VGG16 Loaded Successfully")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
VGG16 Loaded Successfully


In [ ]:
for layer in base_model.layers:
    layer.trainable = False

print("Total Layers :", len(base_model.layers))

Total Layers : 19


In [ ]:
from tensorflow.keras.layers import Flatten,Dense,Dropout
from tensorflow.keras.models import Model

x = base_model.output
x = Flatten()(x)
x = Dense(512,activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(4,activation="softmax")(x)

model = Model(inputs=base_model.input,
              outputs=predictions)

print("Model Created Successfully")

Model Created Successfully


In [ ]:
# ==========================================================
# SECTION 9 : COMPILE MODEL
# ==========================================================

from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("="*60)
print("MODEL COMPILED SUCCESSFULLY")
print("="*60)

MODEL COMPILED SUCCESSFULLY


In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,562,308 (105.14 MB)

 Trainable params: 12,847,620 (49.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
# ==========================================================
# SECTION 11 : AUTO RESUME TRAINING SYSTEM
# ==========================================================

import os
import json
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras.callbacks import ModelCheckpoint, Callback

# -----------------------------
# Random Seed
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# Paths
# -----------------------------
MODEL_PATH = "/content/drive/MyDrive/Research_paper/Models/covid_checkpoint.keras"

STATE_PATH = "/content/drive/MyDrive/Research_paper/Models/training_state.json"

# -----------------------------
# Save Epoch Callback
# -----------------------------
class SaveTrainingState(Callback):

    def on_epoch_end(self, epoch, logs=None):

        state = {
            "last_epoch": epoch + 1
        }

        with open(STATE_PATH, "w") as f:
            json.dump(state, f)

# -----------------------------
# Model Checkpoint
# -----------------------------
checkpoint = ModelCheckpoint(

    MODEL_PATH,

    save_best_only=False,

    save_weights_only=False,

    verbose=1

)

print("="*60)
print("AUTO RESUME SYSTEM READY")
print("="*60)

AUTO RESUME SYSTEM READY


In [35]:
print("TRAIN PATH:", TRAIN_PATH)
print("Training Images:", train_generator.samples)
print("Batch Size:", train_generator.batch_size)
print("Steps Per Epoch:", len(train_generator))

TRAIN PATH: /content/drive/MyDrive/Research_paper/Covid_Data_set/Covid_Split/train
Training Images: 14814
Batch Size: 32
Steps Per Epoch: 463


In [39]:
print("Batch Size :", train_generator.batch_size)
print("Steps Per Epoch :", len(train_generator))
print("Training Images :", train_generator.samples)

Batch Size : 60
Steps Per Epoch : 247
Training Images : 14814
